# TSO-DTF - Paper figures (same style as Wellens, Boute & Udenio 2023)

**How to run:** put `paper_figures_data.zip` in your Drive folder `m5_results` (or upload it when asked), then **Runtime → Run all**.

No training here - it only draws the figures from your final results (about 1 minute). Every figure is shown below and saved as a 300-dpi PNG in `m5_results/paper_figures`.

In [ ]:
from google.colab import drive, files
import os, zipfile
drive.mount("/content/drive")
ZIP = "/content/drive/MyDrive/m5_results/paper_figures_data.zip"
if not os.path.exists(ZIP):
    print("paper_figures_data.zip not found in Drive/m5_results - please upload it now")
    up = files.upload(); ZIP = "/content/" + list(up)[0]
zipfile.ZipFile(ZIP).extractall("/content/figdata")
os.environ["FIG_DATA"] = "/content/figdata"
os.environ["FIG_OUT"] = "/content/drive/MyDrive/m5_results/paper_figures"
print("Data ready:", sorted(os.listdir("/content/figdata")))

## setup

In [ ]:
import os, glob, numpy as np, pandas as pd, matplotlib
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch
DATA = os.environ.get("FIG_DATA", "figdata")
OUT = os.environ.get("FIG_OUT", "paper_figures")
DRIVE_RESULTS = "/content/drive/MyDrive/m5_results"
os.makedirs(OUT, exist_ok=True)
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 300, "font.size": 10, "axes.spines.top": False,
                     "axes.spines.right": False, "axes.grid": True, "grid.color": "#e6e6e6", "grid.linewidth": .7,
                     "axes.axisbelow": True, "axes.edgecolor": "#888", "axes.titleweight": "bold", "axes.titlesize": 11})
BLUE, ORANGE, AQUA, YELLOW, MAGENTA, GREEN, GRAY, RED = "#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#9a9a95", "#d03b3b"
METHOD_COL = {"Baseline DTF": GRAY, "Opt 3": YELLOW, "Opt 4": AQUA, "Opt 5": ORANGE, "Ens 4+5": BLUE}
CAT_COL = {"smooth": BLUE, "erratic": ORANGE, "intermittent": AQUA, "lumpy": MAGENTA}
def save(fig, name):
    fig.savefig(f"{OUT}/{name}.png", bbox_inches="tight"); plt.show(); plt.close(fig)

pp = pd.read_csv(f"{DATA}/per_product.csv")
METHODS = ["Baseline DTF", "Opt 1", "Opt 2", "Opt 3", "Opt 4", "Opt 5", "Ens 3+4", "Ens 4+5", "Ens 3+4+5"]
LABEL = {"Baseline DTF": "Baseline DTF (base paper)", "Opt 1": "Opt 1: CMA-ES + pruning", "Opt 2": "Opt 2: demand pooling",
         "Opt 3": "Opt 3: joint search", "Opt 4": "Opt 4: feature optimization", "Opt 5": "Opt 5: TSO-DTF (two-stage)",
         "Ens 3+4": "Ensemble 3 + 4", "Ens 4+5": "Ensemble 4 + 5 (final)", "Ens 3+4+5": "Ensemble 3 + 4 + 5"}
def wrmsse(d, m): return (d.weight * d["rmsse|" + m]).sum() / d.weight.sum()
def impr(d, m): b = wrmsse(d, "Baseline DTF"); return 100 * (b - wrmsse(d, m)) / b
print(f"{len(pp):,} product-store series loaded")


## Figure 1 - framework

In [ ]:
fig, ax = plt.subplots(figsize=(14, 3.6)); ax.axis("off"); ax.set_xlim(0, 14); ax.set_ylim(0, 3.6)
steps = [("M5 data\n3 stores, 9,119 series\nsales, prices, calendar", "#eef3fa"),
         ("Stage 1\nFeature-group search\n(Optuna TPE, 13 groups)", "#e3f4ec"),
         ("Stage 2\nJoint search:\nhyper-params + pooling\n(CMA-ES, median pruning)", "#fdeee6"),
         ("LightGBM (tweedie)\nper store or per\ndemand pattern", "#eef3fa"),
         ("Ensemble\nOpt 4 + Opt 5\n(final forecast)", "#e3ecfa"),
         ("Evaluation & BI\nRMSSE, SME, WRMSSE,\nWilcoxon, SHAP,\ndashboard", "#f3f3f1")]
w, gap = 2.05, .3
for i, (t, c) in enumerate(steps):
    x = .1 + i * (w + gap)
    ax.add_patch(FancyBboxPatch((x, .9), w, 1.9, boxstyle="round,pad=0.02,rounding_size=0.12", fc=c, ec="#555", lw=1))
    ax.text(x + w / 2, 1.85, t, ha="center", va="center", fontsize=8.8, linespacing=1.35)
    if i < len(steps) - 1:
        ax.annotate("", xy=(x + w + gap - .02, 1.85), xytext=(x + w + .02, 1.85), arrowprops=dict(arrowstyle="-|>", color="#333", lw=1.2))
ax.text(7, .4, "Base paper DTF: fixed manual features + 21 TPE trials per store  →  TSO-DTF optimizes the inputs first, then the model and pooling jointly",
        ha="center", fontsize=9, style="italic", color="#444")
ax.set_title("Figure 1: TSO-DTF - two-stage optimized decision-tree framework", loc="left")
save(fig, "fig1_framework")


## Figure 2 - demand classification

In [ ]:
dc = pd.read_csv(f"{DATA}/demand_pattern_classification.csv").merge(pp[["store_id", "item_id"]])
fig, ax = plt.subplots(figsize=(7.5, 5.2))
for c in ["smooth", "erratic", "intermittent", "lumpy"]:
    s = dc[dc.category == c]
    ax.scatter(s.ADI, s.CV2, s=7, alpha=.55, color=CAT_COL[c], lw=0, label=f"{c.capitalize()} ({len(s):,})")
ax.axvline(4 / 3, color="#333", ls="--", lw=1); ax.axhline(.49, color="#333", ls="--", lw=1)
ax.set_xscale("log"); ax.set_yscale("log"); ax.set_xlabel("ADI (average days between sales)"); ax.set_ylabel("CV²")
ax.legend(markerscale=3, frameon=False, loc="upper right")
ax.set_title(f"Figure 2: Classification of {len(dc):,} product-store series by intermittency (ADI) and variability (CV²)", loc="left", fontsize=10)
save(fig, "fig2_demand_classification")


## Figure 3 - accuracy over time + table

In [ ]:
daily = pd.read_csv(f"{DATA}/daily_rmsse.csv", parse_dates=["date"])
# Baseline line: added automatically if the baseline test predictions file is in your Drive results folder
pred_files = sorted(glob.glob(f"{DRIVE_RESULTS}/test_predictions_*.csv.gz"))
if pred_files:
    tp = pd.read_csv(pred_files[-1], parse_dates=["date"])
    tp = tp[tp.method.str.contains("Baseline", case=False)].merge(pp[["store_id", "item_id", "scale_full"]])
    if len(tp):
        tp = tp.sort_values(["store_id", "item_id", "date"]); tp["e2"] = (tp.sales - tp.pred) ** 2
        tp["r"] = np.sqrt(tp.groupby(["store_id", "item_id"]).e2.transform(lambda s: s.rolling(7).mean()) / tp.scale_full)
        b = tp.dropna(subset=["r"]).groupby("date").r.mean().reset_index().rename(columns={"r": "rmsse_7d"}); b["method"] = "Baseline DTF"
        daily = pd.concat([b, daily]); print("Baseline daily line added from", pred_files[-1])
fig = plt.figure(figsize=(13, 4.6)); ax = fig.add_axes([0.05, 0.14, 0.42, 0.72])
for m in ["Baseline DTF", "Opt 3", "Opt 4", "Opt 5", "Ens 4+5"]:
    d = daily[daily.method == m]
    if len(d): ax.plot(d.date, d.rmsse_7d, color=METHOD_COL[m], lw=2.2 if m == "Ens 4+5" else 1.4, label=LABEL[m])
ax.set_ylabel("RMSSE (7-day rolling)"); ax.set_xlabel("Date (test period 2016)"); ax.legend(frameon=False, fontsize=8, ncol=2, loc="upper left")
ax.tick_params(axis="x", rotation=30)
rows = []
for m in METHODS:
    sm = pp["sme|" + m].mean()
    rows.append([LABEL[m], f"{pp['rmsse|' + m].mean():.4f}", f"{wrmsse(pp, m):.4f}", "-" if m == "Baseline DTF" else f"{impr(pp, m):+.2f}%",
                 "-" if m == "Baseline DTF" else f"{100 * (pp['rmsse|' + m] < pp['rmsse|Baseline DTF']).mean():.1f}%", f"{sm:+.4f}"])
tax = fig.add_axes([0.52, 0.1, 0.47, 0.8]); tax.axis("off")
tb = tax.table(cellText=rows, colLabels=["Method", "RMSSE", "WRMSSE", "Improvement\nover baseline", "Products\nimproved", "SME"], loc="center", cellLoc="center", colWidths=[.34, .11, .11, .15, .13, .11])
tb.auto_set_font_size(False); tb.set_fontsize(8.5); tb.scale(1, 1.55)
for (r, c), cell in tb.get_celld().items():
    cell.set_edgecolor("#ddd")
    if r == 0: cell.set_text_props(weight="bold"); cell.set_facecolor("#f1f1ef")
    if c == 0: cell.set_text_props(ha="left"); cell._loc = "left"
    if r == METHODS.index("Ens 4+5") + 1: cell.set_text_props(weight="bold"); cell.set_facecolor("#e8f0fb")
fig.suptitle("Figure 3: Forecast accuracy of TSO-DTF and the base paper's DTF over the test period (lower is better)", x=0.05, ha="left", fontweight="bold", fontsize=11)
save(fig, "fig3_accuracy_over_time")


## Figure 4 - SHAP contribution pies

In [ ]:
sh = pd.read_csv(f"{DATA}/shap_importance.csv")
def source(f):
    if f.startswith(("sales_lag", "rolling")): return "Sales history"
    if f in ("sell_price",) or f.startswith("price"): return "Price"
    if f in ("snap", "is_event"): return "Promotion & events"
    if f in ("dept_id", "cat_id"): return "Product hierarchy"
    return "Calendar"
RAW = {"sell_price", "snap", "is_event", "dept_id", "cat_id", "day_of_week", "day_of_month", "month", "week_of_year", "quarter"}
sh["source"] = sh.feature.map(source); sh["kind"] = np.where(sh.feature.isin(RAW), "'Raw' inputs", "Engineered inputs")
fig, axs = plt.subplots(1, 2, figsize=(11, 4.8))
a = sh.groupby("source")["share_%"].sum().sort_values(ascending=False)
cols = {"Sales history": "#1c5cab", "Calendar": ORANGE, "Price": YELLOW, "Promotion & events": AQUA, "Product hierarchy": MAGENTA}
axs[0].pie(a, labels=[f"{k}\n{v:.1f}%" if v > 5 else "" for k, v in a.items()], colors=[cols[k] for k in a.index], startangle=90, counterclock=False,
           wedgeprops=dict(edgecolor="white", linewidth=2), textprops=dict(fontsize=9))
axs[0].legend([f"{k} {v:.1f}%" for k, v in a.items()], frameon=False, fontsize=8.5, loc="center left", bbox_to_anchor=(-0.45, 0.5))
axs[0].set_title("By data source")
b = sh.groupby("kind")["share_%"].sum()
axs[1].pie(b, labels=[f"{k}\n{v:.1f}%" for k, v in b.items()], colors=["#1c5cab", "#86b6ef"][:len(b)], startangle=90,
           wedgeprops=dict(edgecolor="white", linewidth=2), textprops=dict(fontsize=9))
axs[1].set_title("Raw vs engineered inputs")
fig.suptitle("Figure 4: Contribution of data sources and feature engineering (share of mean |SHAP value|)", x=0.02, ha="left", fontweight="bold", fontsize=11)
save(fig, "fig4_shap_contribution")


## Figure 5 - top SHAP inputs

In [ ]:
t = sh.head(15).iloc[::-1]
fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(t.feature, t["share_%"], color=[cols[s] for s in t.source], height=.7)
for y, v in enumerate(t["share_%"]): ax.text(v + .2, y, f"{v:.1f}%", va="center", fontsize=8.5)
ax.set_xlabel("Share of total mean |SHAP value| (%)"); ax.grid(axis="y", visible=False)
from matplotlib.patches import Patch
ax.legend(handles=[Patch(color=cols[k], label=k) for k in t.source.unique()], frameon=False, loc="lower right")
ax.set_title("Figure 5: The 15 most important inputs (SHAP, TreeExplainer)", loc="left")
save(fig, "fig5_shap_top_inputs")


## Figure 6 - ablation

In [ ]:
ms = METHODS[1:]; v = [impr(pp, m) for m in ms]
fig, ax = plt.subplots(figsize=(8.5, 4.6))
ax.barh([LABEL[m] for m in ms][::-1], v[::-1], color=[BLUE if x >= 0 else RED for x in v[::-1]], height=.62)
for y, x in enumerate(v[::-1]): ax.text(x + (.02 if x >= 0 else -.02), y, f"{x:+.2f}%", va="center", ha="left" if x >= 0 else "right", fontsize=9)
ax.axvline(0, color="#333", lw=1); ax.set_xlim(min(v) - .25, max(v) + .25); ax.grid(axis="y", visible=False)
ax.set_xlabel("Improvement in WRMSSE over the base paper's DTF (%)")
ax.set_title("Figure 6: Ablation - contribution of each optimization step", loc="left")
save(fig, "fig6_ablation")


## Figure 7 - by demand pattern and store

In [ ]:
show = ["Opt 4", "Opt 5", "Ens 4+5"]; col3 = [AQUA, ORANGE, BLUE]
fig, axs = plt.subplots(1, 2, figsize=(12, 4.3), sharey=True)
for ax, key, groups in [(axs[0], "category", ["smooth", "erratic", "intermittent", "lumpy"]), (axs[1], "store_id", sorted(pp.store_id.unique()))]:
    x = np.arange(len(groups)); wd = .26
    for i, (m, c) in enumerate(zip(show, col3)):
        vals = [impr(pp[pp[key] == g], m) for g in groups]
        ax.bar(x + (i - 1) * wd, vals, wd - .03, color=c, label=LABEL[m])
        for xi, vv in zip(x + (i - 1) * wd, vals): ax.text(xi, vv + .02, f"{vv:.2f}", ha="center", fontsize=7.5)
    ax.set_xticks(x, [g.capitalize() if key == "category" else g for g in groups]); ax.axhline(0, color="#333", lw=1); ax.grid(axis="x", visible=False)
axs[0].set_title("By demand pattern"); axs[1].set_title("By store"); axs[0].set_ylabel("Improvement in WRMSSE (%)")
axs[0].set_ylim(0, 1.1); h, l = axs[0].get_legend_handles_labels(); fig.legend(h, l, frameon=False, fontsize=9, ncol=3, loc="lower center", bbox_to_anchor=(0.5, -0.06))
fig.suptitle("Figure 7: The improvement holds for every demand pattern and every store", x=0.02, ha="left", fontweight="bold", fontsize=11)
save(fig, "fig7_by_pattern_and_store")


## Figure 8 - Opt 4 search convergence

In [ ]:
fs = pd.read_csv(f"{DATA}/optimization4_feature_search.csv"); fs = fs[fs.state == "COMPLETE"].sort_values("number")
fig, ax = plt.subplots(figsize=(8, 4))
ax.scatter(fs.number, fs.value, s=26, facecolor="white", edgecolor=BLUE, lw=1.3, label="Trial", zorder=3)
ax.step(fs.number, fs.value.cummin(), where="post", color=BLUE, lw=2, label="Best so far")
ax.scatter([0], [fs.value.iloc[0]], s=60, color=ORANGE, zorder=4, label="Trial 0 = base paper's features")
ax.set_xlabel("Trial"); ax.set_ylabel("Validation RMSE"); ax.legend(frameon=False)
ax.set_title("Figure 8: Stage 1 feature-group search (Optuna TPE)", loc="left")
save(fig, "fig8_feature_search")


## Figure 9 - forecast vs actual

In [ ]:
ex = pd.read_csv(f"{DATA}/examples.csv", parse_dates=["date"])
keys = ex[["store_id", "item_id"]].drop_duplicates().values
fig, axs = plt.subplots(len(keys), 1, figsize=(11, 2.6 * len(keys)), sharex=True)
for ax, (s, i) in zip(np.atleast_1d(axs), keys):
    d = ex[(ex.store_id == s) & (ex.item_id == i)]
    ax.plot(d.date, d.sales, color="#222", lw=1.3, label="Actual units sold"); ax.plot(d.date, d.pred_45, color=BLUE, lw=1.8, label="TSO-DTF forecast (Ens 4+5)")
    ax.set_title(f"{s} · {i}", loc="left", fontsize=9.5, fontweight="normal"); ax.set_ylabel("Units")
np.atleast_1d(axs)[0].legend(frameon=False, ncol=2, loc="upper left")
fig.suptitle("Figure 9: Daily forecast vs actual sales, best-selling product of each store", x=0.02, ha="left", fontweight="bold", fontsize=11)
fig.tight_layout(); save(fig, "fig9_forecast_vs_actual")


## Figure 10 - per-product change

In [ ]:
chg = 100 * (pp["rmsse|Baseline DTF"] - pp["rmsse|Ens 4+5"]) / pp["rmsse|Baseline DTF"]
fig, ax = plt.subplots(figsize=(8, 4))
c = chg.clip(-15, 15); bins = np.linspace(-15, 15, 61)
ax.hist(c[c < 0], bins=bins, color=RED, alpha=.85, label=f"Worse ({(chg < 0).mean() * 100:.1f}%)")
ax.hist(c[c >= 0], bins=bins, color=BLUE, alpha=.85, label=f"Better ({(chg >= 0).mean() * 100:.1f}%)")
ax.axvline(0, color="#333", lw=1); ax.set_xlabel("RMSSE improvement per product, final vs baseline (%) (clipped at ±15)"); ax.set_ylabel("Products")
ax.legend(frameon=False); ax.set_title("Figure 10: Distribution of per-product improvement", loc="left")
save(fig, "fig10_per_product_change")
print("All figures saved in", OUT)


In [ ]:
import shutil
shutil.make_archive("/content/paper_figures", "zip", os.environ["FIG_OUT"])
files.download("/content/paper_figures.zip")